In [4]:
import pandas as pd

# Load data
constituency_df = pd.read_csv(r"C:\Users\vipul\33_Constituency_Wise_Detailed_Result.csv")
candidates_df = pd.read_csv(r"C:\Users\vipul\Lok_Sabha_2024_candidates_combined.csv")

In [22]:
# Function to convert Indian currency format to numerical
def convert_to_number(value):
    if pd.isna(value) or value == '' or str(value).strip() == 'NaN':
        return None
    value = str(value).strip()
    try:
        # Split by ~ to separate amount from label
        parts = value.split('~')
        if len(parts) < 2:
            return None
        amount_part = parts[0].replace('Rs', '').strip()
        # Remove all commas to get pure number
        amount_str = amount_part.replace(',', '')
        return float(amount_str)
    except:
        return None

In [23]:
# Standardize names
constituency_df['Candidate_Clean'] = constituency_df['Candidate Name'].str.upper().str.strip()
candidates_df['Candidate_Clean'] = candidates_df['Candidate'].str.upper().str.strip()

candidates_df['Constituency_Clean'] = candidates_df['Constituency'].str.upper().str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()
constituency_df['Constituency_Clean'] = constituency_df['PC Name'].str.upper().str.strip()


In [24]:
# Merge
merged_df = constituency_df.merge(
    candidates_df[['Candidate_Clean', 'Constituency_Clean', 'Criminal Cases', 'Education', 'Total Assets', 'Liabilities']],
    on=['Candidate_Clean', 'Constituency_Clean'],
    how='left'
)

In [25]:
# Remove NOTA
merged_df = merged_df[merged_df['Candidate Name'] != 'NOTA'].copy()


In [26]:
# Convert to numerical
merged_df['ASSETS_NUMERIC'] = merged_df['Total Assets'].apply(convert_to_number)
merged_df['LIABILITIES_NUMERIC'] = merged_df['Liabilities'].apply(convert_to_number)


In [27]:
# Create final dataframe
final_df = pd.DataFrame({
    'NAME': merged_df['Candidate Name'],
    'STATE': merged_df['State Name'],
    'CONSTITUENCY': merged_df['PC Name'],
    'PARTY': merged_df['Party Name'],
    'GENDER': merged_df['Gender'],
    'AGE': merged_df['Age'].astype('Int64'),
    'CATEGORY': merged_df['Category'],
    'EDUCATION': merged_df['Education'],
    'CRIMINAL': merged_df['Criminal Cases'].astype('Int64'),
    'ASSETS': merged_df['Total Assets'],
    'ASSETS_NUMERIC': merged_df['ASSETS_NUMERIC'],
    'LIABILITIES': merged_df['Liabilities'],
    'LIABILITIES_NUMERIC': merged_df['LIABILITIES_NUMERIC'],
    'WINNER': (merged_df.groupby('PC Name')['Votes Secured - Total'].rank(method='dense', ascending=False) == 1).astype(int),
    'SYMBOL': merged_df['Party Symbol'],
    'GENERAL': merged_df['Votes Secured - General'].astype('Int64'),
    'POSTAL': merged_df['Votes Secured - Postal'].astype('Int64'),
    'TOTAL': merged_df['Votes Secured - Total'].astype('Int64'),
    '% of Votes Secured - Over Total Votes Polled In Constituency': merged_df['% of Votes Secured - Over Total Votes Polled In Constituency'],
    '% of Votes Secured - Over Total Electors In Constituency': merged_df['% of Votes Secured - Over Total Electors In Constituency'],
    'Over Total Valid Votes Polled In Constituency': merged_df['Over Total Valid Votes Polled In Constituency'],
    'Valid Votes': merged_df['Valid Votes'].astype('Int64'),
    'Total Votes Polled In The Constituency': merged_df['Total Votes Polled In The Constituency'].astype('Int64'),
    'TOTAL ELECTORS': merged_df['Total Electors'].astype('Int64')
})


In [28]:

# Save
final_df.to_csv('Election-data.csv', index=False)